In [2]:
from transformers import AutoTokenizer,AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch
from peft import PeftModel
import pandas as pd
from tqdm import tqdm
import re
import time
from prepare_data_for_dpo import generate_batch
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
HF_TOKEN=os.getenv('HF_TOKEN')
model_name = "Qwen/Qwen3-4B"
tokenizer=AutoTokenizer.from_pretrained(model_name,token=HF_TOKEN)
tokenizer.pad_token=tokenizer.eos_token

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [6]:
bnb_config_base_model=BitsAndBytesConfig(

    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [7]:
base_model=AutoModelForCausalLM.from_pretrained(model_name,
                                                device_map='auto',
                                                token=HF_TOKEN,
                                                quantization_config=bnb_config_base_model,
                                                dtype=torch.float16)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [8]:
base_model_for_sft=AutoModelForCausalLM.from_pretrained(model_name,
                                                device_map='auto',
                                                token=HF_TOKEN,
                                                quantization_config=bnb_config_base_model,
                                                dtype=torch.float16)
sft_model=PeftModel.from_pretrained(base_model_for_sft,'sft_model')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [9]:
base_model_for_dpo_without_sft_answers=AutoModelForCausalLM.from_pretrained(model_name,
                                                device_map='auto',
                                                token=HF_TOKEN,
                                                quantization_config=bnb_config_base_model,
                                                dtype=torch.float16)
dpo_model_without_sft_answers=PeftModel.from_pretrained(base_model_for_dpo_without_sft_answers,'dpo_model_without_sft_answers')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [10]:
base_model_for_dpo_extended=AutoModelForCausalLM.from_pretrained(model_name,
                                                device_map='auto',
                                                token=HF_TOKEN,
                                                quantization_config=bnb_config_base_model,
                                                dtype=torch.float16)
dpo_model_extended=PeftModel.from_pretrained(base_model_for_dpo_extended,'dpo_model_extended')

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
test_data=pd.read_csv('final_test_data/test_data_final_all_models.csv',index_col=0)
test_data.info()
test_data=test_data.sample(1000)

<class 'pandas.DataFrame'>
Index: 1000 entries, 999 to 803
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Unnamed: 0    1000 non-null   int64
 1   text          1000 non-null   str  
 2   from_dataset  1000 non-null   str  
 3   is_unsafe     1000 non-null   int64
 4   user_message  1000 non-null   str  
dtypes: int64(2), str(3)
memory usage: 1.8 MB


In [5]:
test_data['is_unsafe'] = test_data['text'].apply(lambda x:
    int(re.search(r'is_unsafe:\s*(\d+)', str(x)).group(1))
    if re.search(r'is_unsafe:\s*(\d+)', str(x)) else None
)
test_data['user_message'] = test_data['text'].str.extract(
    r'<\|im_start\|>user\n(.*?)<\|im_end\|>',
    expand=False,
    flags=re.DOTALL
).str.strip()
list_prompts=test_data['user_message'].tolist()

In [16]:
test_data.to_csv('test_data_final_all_models.csv')

In [6]:
test_data.info()

<class 'pandas.DataFrame'>
Index: 1000 entries, 855 to 849
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Unnamed: 0    1000 non-null   int64
 1   text          1000 non-null   str  
 2   from_dataset  1000 non-null   str  
 3   is_unsafe     1000 non-null   int64
 4   user_message  1000 non-null   str  
dtypes: int64(2), str(3)
memory usage: 1.8 MB


In [13]:
CONFIG={'do_sample':False, 'repetition_penalty':1.1}

In [25]:
responses_base_model=generate_batch(model=base_model, tokenizer=tokenizer, prompts=list_prompts, config=CONFIG, batch_size=8, max_new_tokens=550)
data_base=pd.Series(responses_base_model)
data_base.to_csv('responses_base_model.csv')

100%|██████████| 125/125 [2:48:37<00:00, 80.94s/it]

Total time: 10117.16 seconds
Average time per prompt: 10.12 seconds
Prompts per second: 0.10


In [17]:
responses_sft_model=generate_batch(model=sft_model, tokenizer=tokenizer, prompts=list_prompts, config=CONFIG, batch_size=8, max_new_tokens=550)
data_sft=pd.Series(responses_sft_model)
data_sft.to_csv('responses_sft_model_final.csv')

100%|██████████| 6/6 [05:03<00:00, 50.65s/it]

Total time: 303.93 seconds
Average time per prompt: 7.24 seconds
Prompts per second: 0.14


In [18]:
responses_dpo_model_without_sft_answers_model=generate_batch(model=dpo_model_without_sft_answers, tokenizer=tokenizer, prompts=list_prompts, config=CONFIG, batch_size=8, max_new_tokens=550)
data_dpo_model_without_sft_answers=pd.Series(responses_dpo_model_without_sft_answers_model)
data_dpo_model_without_sft_answers.to_csv('responses_dpo_model_without_sft_answers_final.csv')

100%|██████████| 6/6 [04:37<00:00, 46.21s/it]

Total time: 277.23 seconds
Average time per prompt: 6.60 seconds
Prompts per second: 0.15


In [19]:
responses_dpo_extended_model=generate_batch(model=dpo_model_extended, tokenizer=tokenizer, prompts=list_prompts, config=CONFIG, batch_size=8, max_new_tokens=550)
data_dpo_extended_model=pd.Series(responses_dpo_extended_model)
data_dpo_extended_model.to_csv('responses_dpo_extended_model.csv')

100%|██████████| 6/6 [06:18<00:00, 63.04s/it]

Total time: 378.22 seconds
Average time per prompt: 9.01 seconds
Prompts per second: 0.11
